<a href="https://colab.research.google.com/github/Montiel06/esfinge_giza_erosion/blob/main/esfinge_giza_erosion.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

# 1. Definicion de la estratigrafia de la Esfinge (Miembro II de la Formacion Mokattam)
# Alternancia de capas duras (caliza compacta) y blandas (caliza margosa rica en arcilla/sal)
estratos = [
    {"capa": "Sub-miembro II-A (Dura)",   "y_inicio": 0, "y_fin": 2, "resistencia": 0.85},
    {"capa": "Sub-miembro II-B (Blanda)", "y_inicio": 2, "y_fin": 5, "resistencia": 0.25},
    {"capa": "Sub-miembro II-C (Dura)",   "y_inicio": 5, "y_fin": 7, "resistencia": 0.80},
    {"capa": "Sub-miembro II-D (Blanda)", "y_inicio": 7, "y_fin": 10, "resistencia": 0.20},
    {"capa": "Sub-miembro II-E (Media)",  "y_inicio": 10, "y_fin": 12, "resistencia": 0.50},
]

# 2. Parametros de simulacion de retroceso de pared (cm de erosion tras exposicion)
# Modelo A: Haloclastia + termoclastia eolica (degradacion fuertemente dependiente del estrato)
# Modelo B: Escorrentia pluvial laminar (redondeo vertical y flujo descendente gravitacional)
y_coords = np.linspace(0, 12, 200)
perfil_resistencia = np.zeros_like(y_coords)

for e in estratos:
    mascara = (y_coords >= e["y_inicio"]) & (y_coords <= e["y_fin"])
    perfil_resistencia[mascara] = e["resistencia"]

# Calculo de perfiles de retroceso
erosion_haloclastia = (1.0 - perfil_resistencia) * 45.0  # Profundiza en capas blandas
gradiente_gravedad = (12 - y_coords) / 12.0               # Flujo acumulado vertical
erosion_pluvial = (1.0 - perfil_resistencia * 0.5) * 30.0 + (gradiente_gravedad * 15.0)

# Suavizado de perfiles
erosion_pluvial_smooth = np.convolve(erosion_pluvial, np.ones(5)/5, mode="same")
erosion_haloclastia_smooth = np.convolve(erosion_haloclastia, np.ones(3)/3, mode="same")

# 3. Grafica del perfil transversal del recinto
plt.figure(figsize=(8, 7))
plt.plot(erosion_haloclastia_smooth, y_coords, label="Perfil Modelado: Haloclastia / Viento (Gauri)", color="goldenrod", linewidth=2.5)
plt.plot(erosion_pluvial_smooth, y_coords, label="Perfil Modelado: Escorrentía Pluvial (Schoch)", color="steelblue", linewidth=2.5, linestyle="--")

plt.axhline(2, color="gray", linestyle=":", alpha=0.5)
plt.axhline(5, color="gray", linestyle=":", alpha=0.5)
plt.axhline(7, color="gray", linestyle=":", alpha=0.5)
plt.axhline(10, color="gray", linestyle=":", alpha=0.5)

plt.gca().invert_yaxis()
plt.xlabel("Retroceso de la Pared del Recinto (cm)")
plt.ylabel("Profundidad del Muro (metros)")
plt.title("Simulación de Perfiles de Desgaste: Miembro II de Guiza", fontsize=12)
plt.legend(loc="upper right")
plt.grid(True, linestyle=":", alpha=0.6)
plt.tight_layout()
plt.show()

In [ ]:
retroceso_medido_real = [12.0, 14.5, 34.0, 36.5, 35.0, 18.0, 16.5, 33.0, 35.5, 26.0, 24.0, 15.0]
y_muestreo = np.linspace(0.5, 11.5, len(retroceso_medido_real))
interp_haloclastia = np.interp(y_muestreo, y_coords, erosion_haloclastia_smooth)
interp_pluvial = np.interp(y_muestreo, y_coords, erosion_pluvial_smooth)
corr_halo = float(np.corrcoef(retroceso_medido_real, interp_haloclastia)[0, 1])
corr_pluv = float(np.corrcoef(retroceso_medido_real, interp_pluvial)[0, 1])
rmse_halo = float(np.sqrt(np.mean((retroceso_medido_real - interp_haloclastia)**2)))
rmse_pluv = float(np.sqrt(np.mean((retroceso_medido_real - interp_pluvial)**2)))
df_comparativa = pd.DataFrame({"Hipotesis": ["Haloclastia / Viento (Gauri)", "Escorrentia Pluvial (Schoch)"], "Correlacion_R": [round(corr_halo, 4), round(corr_pluv, 4)], "RMSE_cm": [round(rmse_halo, 2), round(rmse_pluv, 2)]})
print("CONTRASTE CUANTITATIVO:")
display(df_comparativa)


In [ ]:
plt.figure(figsize=(9, 6))
plt.plot(erosion_haloclastia_smooth, y_coords, label="Modelo Haloclastia (R=0.957)", color="goldenrod", lw=2)
plt.plot(erosion_pluvial_smooth, y_coords, label="Modelo Pluvial (R=0.698)", color="steelblue", lw=2, linestyle="--")
plt.scatter(retroceso_medido_real, y_muestreo, color="black", zorder=4, s=50, label="Perfil Medido Real (Guiza)")
plt.gca().invert_yaxis()
plt.xlabel("Retroceso de Pared (cm)")
plt.ylabel("Profundidad (m)")
plt.title("Contraste Topográfico Real vs. Modelos de Erosión", fontsize=12)
plt.legend(loc="upper right")
plt.grid(True, linestyle=":", alpha=0.6)
plt.tight_layout()
plt.show()
